This corresponds to task e) in the problem set: 

In [ ]:
#IMPORTS: 
from tensorflow.keras.datasets import mnist
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import jax.numpy as jnp
import numpy as np
import neural_network_functions as nnf
import matplotlib.pyplot as plt
import time

import importlib
importlib.reload(nnf)

print(hasattr(nnf, "classification_accuracy"))


In [ ]:

#Huge MNIST dataset: 
(X_train, y_train), (X_test, y_test) = mnist.load_data()
y_train = np.eye(10, dtype=np.float32)[y_train]
y_test = np.eye(10, dtype=np.float32)[y_test]
print(X_train.shape) # (60000, 28, 28)
print(y_train.shape) # (60000,)
print(X_test.shape)
print(y_test.shape)

#Smaller digits dataset used for testing: 
digits = load_digits()
X = digits.data / 16      # shape (1797, 64)
y = digits.target.astype(int) # labels 0-9
X_digits_train, X_digits_test, y_digits_train, y_digits_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)
print(X_digits_train.shape)
print(y_digits_train.shape)

Prediction accuracy of digits dataset before training

In [ ]:
##Written by copilot: 
network_input_size = 64
layer_output_sizes = [32, 10]

activation_funcs = [nnf.sigmoid, nnf.sigmoid]

layers = nnf.create_layers(
    network_input_size,
    layer_output_sizes,
)

# Predictions from untrained network
outputs = nnf.feed_forward(
    X_digits_test,
    layers,
    activation_funcs,
)

# Predicted digit = index of largest output neuron
y_pred = np.argmax(outputs, axis=1)

accuracy = np.mean(y_pred == y_digits_test)

print(f"Accuracy before training: {accuracy:.4f}")

Fixing ADAM as optimizer and network size as [784,100,100,10], and looking at accuracy for different activation functions and initialization methods on MNIST data. 

NOTE: Will take 30 minutes to run!

Equivalent test for digits data. Runs much faster 

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt


# ---------------------------------------------------------
# Experiment configuration
# ---------------------------------------------------------

network_input_size = 64
layer_output_sizes = [100, 100, 10]

epochs = 20
batch_size = 128
learning_rate = 0.001

initialization_methods = [
    "LeCun",
    "Xavier",
    "He",
]

activation_experiments = {
    "Sigmoid": (
        nnf.sigmoid,
        nnf.sigmoid_der,
    ),
    "ReLU": (
        nnf.ReLU,
        nnf.ReLU_der,
    ),
    "Leaky ReLU": (
        nnf.leaky_ReLU,
        nnf.leaky_ReLU_der,
    ),
}

results_digits = []
trained_models_digits = {}


# ---------------------------------------------------------
# Prepare digits data
# ---------------------------------------------------------

X_digits_train = np.asarray(
    X_digits_train,
    dtype=np.float64,
)

X_digits_test = np.asarray(
    X_digits_test,
    dtype=np.float64,
)

# Ensure integer labels have shape (n_samples,)
y_digits_train = np.asarray(
    y_digits_train
).ravel().astype(int)

y_digits_test = np.asarray(
    y_digits_test
).ravel().astype(int)

# One-hot labels used during MSE training
Y_digits_train_onehot = np.eye(10)[y_digits_train]
Y_digits_test_onehot = np.eye(10)[y_digits_test]

print(
    "Training data range:",
    X_digits_train.min(),
    X_digits_train.max(),
)

print("Training shape:", X_digits_train.shape)
print("Test shape:", X_digits_test.shape)


# ---------------------------------------------------------
# Run experiments
# ---------------------------------------------------------

for initialization_method in initialization_methods:

    print("\n" + "=" * 60)
    print(f"Initialization: {initialization_method}")
    print("=" * 60)

    # Use a repeatable initialization for this method
    np.random.seed(42)

    initial_layers_digits = nnf.create_layers(
        network_input_size,
        layer_output_sizes,
        method=initialization_method,
    )

    for activation_name, (
        hidden_activation,
        hidden_derivative,
    ) in activation_experiments.items():

        print(
            f"\nTraining with {activation_name} "
            f"and {initialization_method} initialization..."
        )

        # Each activation starts with the same parameters
        # for the current initialization method
        layers = [
            (W.copy(), b.copy())
            for W, b in initial_layers_digits
        ]

        activation_funcs = [
            hidden_activation,
            hidden_activation,
            nnf.softmax,
        ]

        activation_ders = [
            hidden_derivative,
            hidden_derivative,
            None,
        ]

        accuracy_before = nnf.classification_accuracy(
            layers,
            X_digits_test,
            y_digits_test,
            activation_funcs,
        )

        # Reset the random seed so each experiment gets the
        # same mini-batch shuffle order
        np.random.seed(123)

        start_time = time.perf_counter()

        layers = nnf.train_sgd_adam(
            Xtrain=X_digits_train,
            Ytrain=Y_digits_train_onehot,
            layers=layers,
            activation_funcs=activation_funcs,
            activation_ders=activation_ders,
            epochs=epochs,
            batch_size=batch_size,
            gamma=learning_rate,
            beta1=0.9,
            beta2=0.999,
            use_decaying_step_length=False,
            cost_der=nnf.cross_entropy_der,
        )

        training_time = time.perf_counter() - start_time

        train_accuracy = nnf.classification_accuracy(
            layers,
            X_digits_train,
            y_digits_train,
            activation_funcs,
        )

        test_accuracy = nnf.classification_accuracy(
            layers,
            X_digits_test,
            y_digits_test,
            activation_funcs,
        )

        # Inspect the trained pre-activation values
        layer_inputs, zs, outputs = nnf.feed_forward_saver(
            X_digits_train,
            layers,
            activation_funcs,
        )

        layer_statistics = []

        for layer_number, z in enumerate(zs, start=1):
            z = np.asarray(z)

            layer_statistics.append({
                "Layer": layer_number,
                "Mean": float(z.mean()),
                "Standard deviation": float(z.std()),
                "Fraction <= 0": float(np.mean(z <= 0)),
            })
            print(f"\nHidden layer {layer_number}")
            print(f"z mean:               {z.mean():.4f}")
            print(f"z standard deviation: {z.std():.4f}")
            print(f"z minimum:            {z.min():.4f}")
            print(f"z maximum:            {z.max():.4f}")

        result = {
            "Initialization": initialization_method,
            "Activation": activation_name,
            "Accuracy before training": accuracy_before,
            "Training accuracy": train_accuracy,
            "Test accuracy": test_accuracy,
            "Training time (seconds)": training_time,
        }

        results_digits.append(result)

        # Use both settings as the model key
        model_key = (
            initialization_method,
            activation_name,
        )

        trained_models_digits[model_key] = {
            "layers": layers,
            "activation_funcs": activation_funcs,
            "activation_ders": activation_ders,
            "layer_statistics": layer_statistics,
        }

        print(f"Before training:   {accuracy_before:.4f}")
        print(f"Training accuracy: {train_accuracy:.4f}")
        print(f"Test accuracy:     {test_accuracy:.4f}")
        print(f"Training time:     {training_time:.1f} seconds")

        for statistics in layer_statistics:
            print(
                f"Layer {statistics['Layer']}: "
                f"mean={statistics['Mean']:.4f}, "
                f"std={statistics['Standard deviation']:.4f}, "
                f"fraction <= 0="
                f"{statistics['Fraction <= 0']:.4f}"
            )

Results: 

In [ ]:
activation_names = list(activation_experiments.keys())

train_accuracy_matrix = np.zeros(
    (
        len(activation_names),
        len(initialization_methods),
    )
)

test_accuracy_matrix = np.zeros_like(
    train_accuracy_matrix
)

initial_accuracy_matrix = np.zeros_like(
    train_accuracy_matrix
)

training_time_matrix = np.zeros_like(
    train_accuracy_matrix
)

for result in results_digits:
    activation_index = activation_names.index(
        result["Activation"]
    )

    initialization_index = initialization_methods.index(
        result["Initialization"]
    )

    train_accuracy_matrix[
        activation_index,
        initialization_index,
    ] = result["Training accuracy"]

    test_accuracy_matrix[
        activation_index,
        initialization_index,
    ] = result["Test accuracy"]

    initial_accuracy_matrix[
        activation_index,
        initialization_index,
    ] = result["Accuracy before training"]

    training_time_matrix[
        activation_index,
        initialization_index,
    ] = result["Training time (seconds)"]



print(
    "Predicted class counts:",
    np.bincount(y_pred, minlength=10),
)

print(
    "True class counts:",
    np.bincount(y_digits_test, minlength=10),
)

In [ ]:
x = np.arange(len(activation_names))
bar_width = 0.24

fig, ax = plt.subplots(figsize=(10, 6))

colors = [
    "tab:blue",
    "tab:orange",
    "tab:green",
]

for initialization_index, initialization_method in enumerate(
    initialization_methods
):
    positions = (
        x
        + (
            initialization_index
            - (len(initialization_methods) - 1) / 2
        )
        * bar_width
    )

    bars = ax.bar(
        positions,
        test_accuracy_matrix[:, initialization_index],
        width=bar_width,
        label=initialization_method,
        color=colors[initialization_index],
        edgecolor="black",
        linewidth=0.5,
    )

    ax.bar_label(
        bars,
        fmt="%.3f",
        padding=3,
        fontsize=9,
    )

ax.axhline(
    0.10,
    color="gray",
    linestyle="--",
    linewidth=1.2,
    label="Random guessing",
)

ax.set_title(
    "Digits test accuracy by activation and initialization"
)

ax.set_xlabel("Hidden-layer activation function")
ax.set_ylabel("Test accuracy")

ax.set_xticks(x)
ax.set_xticklabels(activation_names)

ax.set_ylim(
    0,
    min(
        1.05,
        max(0.2, test_accuracy_matrix.max() + 0.12),
    ),
)

ax.legend(
    title="Weight initialization",
    loc="best",
)

ax.grid(
    axis="y",
    alpha=0.25,
)

fig.tight_layout()
plt.show()

Fixing Leaky ReLU with He initialization, and varying omtimization method for the same model architecture. 

In [ ]:
# ---------------------------------------------------------
# Experiment configuration
# ---------------------------------------------------------

network_input_size = 64
layer_output_sizes = [200, 200, 10]

epochs = 20
batch_size = 128
learning_rate = 0.001

optimizer_names = [
    "Plain SGD",
    "RMSprop",
    "Adam",
]

optimizer_results = []
trained_optimizer_models = {}


# ---------------------------------------------------------
# Prepare the digits data
# ---------------------------------------------------------

X_digits_train = np.asarray(
    X_digits_train,
    dtype=np.float64,
)

X_digits_test = np.asarray(
    X_digits_test,
    dtype=np.float64,
)

# Avoid repeatedly scaling the data when rerunning this cell
if X_digits_train.max() > 1.0:
    X_digits_train = X_digits_train / 16.0
    X_digits_test = X_digits_test / 16.0

y_digits_train = np.asarray(
    y_digits_train
).ravel().astype(int)

y_digits_test = np.asarray(
    y_digits_test
).ravel().astype(int)

# One-hot targets used with categorical cross-entropy
Y_digits_train_onehot = np.eye(10)[y_digits_train]
Y_digits_test_onehot = np.eye(10)[y_digits_test]


# ---------------------------------------------------------
# Fixed activation functions
# ---------------------------------------------------------

# Two Leaky ReLU hidden layers and one softmax output layer
activation_funcs = [
    nnf.leaky_ReLU,
    nnf.leaky_ReLU,
    nnf.softmax,
]

activation_ders = [
    nnf.leaky_ReLU_der,
    nnf.leaky_ReLU_der,
    None,  # Combined softmax + cross-entropy derivative
]


# ---------------------------------------------------------
# Create one shared He initialization
# ---------------------------------------------------------

np.random.seed(42)

initial_layers_digits = nnf.create_layers(
    network_input_size,
    layer_output_sizes,
    method="He",
)


# ---------------------------------------------------------
# Run optimizer experiments
# ---------------------------------------------------------

for optimizer_name in optimizer_names:
    print("\n" + "=" * 60)
    print(f"Training with {optimizer_name}")
    print("=" * 60)

    # Give every optimizer identical initial parameters
    layers = [
        (W.copy(), b.copy())
        for W, b in initial_layers_digits
    ]

    accuracy_before = nnf.classification_accuracy(
        layers,
        X_digits_test,
        y_digits_test,
        activation_funcs,
    )

    # Give every optimizer the same mini-batch shuffle sequence
    np.random.seed(123)

    start_time = time.perf_counter()

    if optimizer_name == "Plain SGD":
        layers = nnf.train_sgd_plain(
            Xtrain=X_digits_train,
            Ytrain=Y_digits_train_onehot,
            layers=layers,
            activation_funcs=activation_funcs,
            activation_ders=activation_ders,
            epochs=epochs,
            batch_size=batch_size,
            gamma=learning_rate,
            use_decaying_step_length=False,
            cost_der=nnf.cross_entropy_der,
        )

    elif optimizer_name == "RMSprop":
        layers = nnf.train_sgd_RMSprop(
            Xtrain=X_digits_train,
            Ytrain=Y_digits_train_onehot,
            layers=layers,
            activation_funcs=activation_funcs,
            activation_ders=activation_ders,
            epochs=epochs,
            batch_size=batch_size,
            gamma=learning_rate,
            beta2=0.999,
            use_decaying_step_length=False,
            cost_der=nnf.cross_entropy_der,
        )

    elif optimizer_name == "Adam":
        layers = nnf.train_sgd_adam(
            Xtrain=X_digits_train,
            Ytrain=Y_digits_train_onehot,
            layers=layers,
            activation_funcs=activation_funcs,
            activation_ders=activation_ders,
            epochs=epochs,
            batch_size=batch_size,
            gamma=learning_rate,
            beta1=0.9,
            beta2=0.999,
            use_decaying_step_length=False,
            cost_der=nnf.cross_entropy_der,
        )

    training_time = time.perf_counter() - start_time

    train_accuracy = nnf.classification_accuracy(
        layers,
        X_digits_train,
        y_digits_train,
        activation_funcs,
    )

    test_accuracy = nnf.classification_accuracy(
        layers,
        X_digits_test,
        y_digits_test,
        activation_funcs,
    )

    optimizer_results.append({
        "Optimizer": optimizer_name,
        "Accuracy before training": accuracy_before,
        "Training accuracy": train_accuracy,
        "Test accuracy": test_accuracy,
        "Training time": training_time,
    })

    trained_optimizer_models[optimizer_name] = {
        "layers": layers,
        "activation_funcs": activation_funcs,
    }

    print(f"Before training:   {accuracy_before:.4f}")
    print(f"Training accuracy: {train_accuracy:.4f}")
    print(f"Test accuracy:     {test_accuracy:.4f}")
    print(f"Training time:     {training_time:.2f} seconds")

In [ ]:
# ---------------------------------------------------------
# Extract results
# ---------------------------------------------------------

optimizer_labels = [
    result["Optimizer"]
    for result in optimizer_results
]

test_accuracy = np.array([
    result["Test accuracy"]
    for result in optimizer_results
])

training_time = np.array([
    result["Training time"]
    for result in optimizer_results
])


# ---------------------------------------------------------
# Plot configuration
# ---------------------------------------------------------

x = np.arange(len(optimizer_labels))

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5.5),
)


# ---------------------------------------------------------
# Plot 1: Test accuracy
# ---------------------------------------------------------

test_bars = axes[0].bar(
    x,
    test_accuracy,
    width=0.55,
    color="tab:orange",
    edgecolor="black",
    linewidth=0.6,
)

# Random-guessing baseline for 10 classes
axes[0].axhline(
    y=0.10,
    color="gray",
    linestyle="--",
    linewidth=1.2,
    label="Random guessing",
)

axes[0].set_title("Test accuracy by optimization method for [200, 200, 10] NN")
axes[0].set_xlabel("Optimization method")
axes[0].set_ylabel("Test accuracy")

axes[0].set_xticks(x)
axes[0].set_xticklabels(optimizer_labels)
axes[0].set_ylim(0, 1.08)

axes[0].grid(
    axis="y",
    alpha=0.25,
)

axes[0].legend()

axes[0].bar_label(
    test_bars,
    fmt="%.3f",
    padding=3,
)


# ---------------------------------------------------------
# Plot 2: Training time
# ---------------------------------------------------------

time_bars = axes[1].bar(
    x,
    training_time,
    width=0.55,
    color="tab:green",
    edgecolor="black",
    linewidth=0.6,
)

axes[1].set_title("Training time by optimization method")
axes[1].set_xlabel("Optimization method")
axes[1].set_ylabel("Training time (seconds)")

axes[1].set_xticks(x)
axes[1].set_xticklabels(optimizer_labels)

axes[1].grid(
    axis="y",
    alpha=0.25,
)

axes[1].bar_label(
    time_bars,
    labels=[
        f"{time_value:.2f} s"
        for time_value in training_time
    ],
    padding=3,
)


# ---------------------------------------------------------
# Layout
# ---------------------------------------------------------

fig.tight_layout()
plt.show()

Since RMSProp got highest accuracy and lowest training time, we fix RMSprop.

Now we analyze varying model architecture. 
We fix
- Leaky ReLU
- He initialization
- RMS prop optimization


We look at the following types of architectures, fixing 400 hidden nodes per model exept logistic:
- [64, 10]  (Logistic)
- [64, 400, 10]
- [64, 50, 125, 225, 10]
- [64, 225, 125, 50, 10]
- [64, 100, 200, 100, 10]
- [64, 200, 10, 200, 10]
- [64, 50, 50, 50, 50, 50, 50, 50, 50, 10]

In [ ]:

# ---------------------------------------------------------
# Architecture configurations
# ---------------------------------------------------------

network_input_size = 64

architecture_experiments = {
    "Logistic regression": [10],
    "Single wide layer": [400, 10],
    "Increasing": [50, 125, 225, 10],
    "Decreasing": [225, 125, 50, 10],
    "Small-big-small": [100, 200, 100, 10],
    "Big-small-big": [200, 10, 200, 10],
    "Constant deep": [50, 50, 50, 50, 50, 50, 50, 50, 10],
}


# ---------------------------------------------------------
# Fixed training configuration
# ---------------------------------------------------------

epochs = 20
batch_size = 128
learning_rate = 0.001

architecture_results = []
trained_architecture_models = {}


# ---------------------------------------------------------
# Prepare data
# ---------------------------------------------------------

X_digits_train = np.asarray(
    X_digits_train,
    dtype=np.float64,
)

X_digits_test = np.asarray(
    X_digits_test,
    dtype=np.float64,
)

# Prevent scaling the data repeatedly when rerunning the cell
if X_digits_train.max() > 1.0:
    X_digits_train = X_digits_train / 16.0
    X_digits_test = X_digits_test / 16.0

y_digits_train = np.asarray(
    y_digits_train
).ravel().astype(int)

y_digits_test = np.asarray(
    y_digits_test
).ravel().astype(int)

# One-hot targets for softmax and cross-entropy training
Y_digits_train_onehot = np.eye(10)[y_digits_train]
Y_digits_test_onehot = np.eye(10)[y_digits_test]

print(
    "Training input range:",
    X_digits_train.min(),
    "to",
    X_digits_train.max(),
)

print("Training data shape:", X_digits_train.shape)
print("Training target shape:", Y_digits_train_onehot.shape)


# ---------------------------------------------------------
# Run the architecture experiments
# ---------------------------------------------------------

for architecture_name, layer_output_sizes in architecture_experiments.items():

    full_architecture = [
        network_input_size,
        *layer_output_sizes,
    ]

    print("\n" + "=" * 65)
    print(f"Architecture: {architecture_name}")
    print(
        "Network:",
        " -> ".join(map(str, full_architecture)),
    )
    print("=" * 65)

    # Same seed for reproducible initialization
    np.random.seed(42)

    # He initialization for all layers
    layers = nnf.create_layers(
        network_input_size,
        layer_output_sizes,
        method="He",
    )

    number_of_hidden_layers = len(layer_output_sizes) - 1

    # Leaky ReLU for every hidden layer
    # Softmax for the output layer
    activation_funcs = (
        [nnf.leaky_ReLU] * number_of_hidden_layers
        + [nnf.softmax]
    )

    activation_ders = (
        [nnf.leaky_ReLU_der] * number_of_hidden_layers
        + [None]
    )

    accuracy_before = nnf.classification_accuracy(
        layers,
        X_digits_test,
        y_digits_test,
        activation_funcs,
    )

    # Use the same mini-batch shuffle sequence
    np.random.seed(123)

    start_time = time.perf_counter()

    layers = nnf.train_sgd_RMSprop(
        Xtrain=X_digits_train,
        Ytrain=Y_digits_train_onehot,
        layers=layers,
        activation_funcs=activation_funcs,
        activation_ders=activation_ders,
        epochs=epochs,
        batch_size=batch_size,
        gamma=learning_rate,
        beta2=0.999,
        use_decaying_step_length=False,
        cost_der=nnf.cross_entropy_der,
    )

    training_time = time.perf_counter() - start_time

    training_accuracy = nnf.classification_accuracy(
        layers,
        X_digits_train,
        y_digits_train,
        activation_funcs,
    )

    test_accuracy = nnf.classification_accuracy(
        layers,
        X_digits_test,
        y_digits_test,
        activation_funcs,
    )

    # Total number of weights and biases
    parameter_count = sum(
        W.size + b.size
        for W, b in layers
    )

    architecture_results.append({
        "Architecture": architecture_name,
        "Layer sizes": full_architecture,
        "Hidden layers": number_of_hidden_layers,
        "Parameters": parameter_count,
        "Accuracy before training": accuracy_before,
        "Training accuracy": training_accuracy,
        "Test accuracy": test_accuracy,
        "Training time": training_time,
    })

    trained_architecture_models[architecture_name] = {
        "layers": layers,
        "activation_funcs": activation_funcs,
        "activation_ders": activation_ders,
    }

    print(f"Hidden layers:       {number_of_hidden_layers}")
    print(f"Parameters:          {parameter_count:,}")
    print(f"Before training:     {accuracy_before:.4f}")
    print(f"Training accuracy:   {training_accuracy:.4f}")
    print(f"Test accuracy:       {test_accuracy:.4f}")
    print(f"Training time:       {training_time:.2f} seconds")

In [ ]:
# ---------------------------------------------------------
# Extract plotting values
# ---------------------------------------------------------

architecture_labels = [
    result["Architecture"]
    for result in architecture_results
]

test_accuracies = np.array([
    result["Test accuracy"]
    for result in architecture_results
])

training_times = np.array([
    result["Training time"]
    for result in architecture_results
])


# ---------------------------------------------------------
# Create plots
# ---------------------------------------------------------

x = np.arange(len(architecture_labels))

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 6),
)


# ---------------------------------------------------------
# Plot 1: Test accuracy
# ---------------------------------------------------------

test_bars = axes[0].bar(
    x,
    test_accuracies,
    width=0.6,
    color="tab:orange",
    edgecolor="black",
    linewidth=0.6,
)

axes[0].axhline(
    0.10,
    color="gray",
    linestyle="--",
    linewidth=1.2,
    label="Random guessing",
)

axes[0].set_title("Test accuracy by architecture")
axes[0].set_xlabel("Architecture")
axes[0].set_ylabel("Test accuracy")

axes[0].set_xticks(x)
axes[0].set_xticklabels(
    architecture_labels,
    rotation=35,
    ha="right",
)

axes[0].set_ylim(0, 1.08)
axes[0].grid(axis="y", alpha=0.25)
axes[0].legend()

axes[0].bar_label(
    test_bars,
    fmt="%.3f",
    padding=3,
    fontsize=9,
)


# ---------------------------------------------------------
# Plot 2: Training runtime
# ---------------------------------------------------------

time_bars = axes[1].bar(
    x,
    training_times,
    width=0.6,
    color="tab:green",
    edgecolor="black",
    linewidth=0.6,
)

axes[1].set_title("Training runtime by architecture")
axes[1].set_xlabel("Architecture")
axes[1].set_ylabel("Training time (seconds)")

axes[1].set_xticks(x)
axes[1].set_xticklabels(
    architecture_labels,
    rotation=35,
    ha="right",
)

axes[1].grid(axis="y", alpha=0.25)

axes[1].bar_label(
    time_bars,
    labels=[
        f"{training_time:.2f} s"
        for training_time in training_times
    ],
    padding=3,
    fontsize=9,
)

# Add a little vertical space for the runtime labels
axes[1].set_ylim(
    0,
    training_times.max() * 1.15,
)


# ---------------------------------------------------------
# Overall title and layout
# ---------------------------------------------------------

fig.suptitle(
    "Architecture comparison on the digits dataset\n"
    "He initialization, RMSprop, softmax and cross-entropy; "
    "Leaky ReLU in hidden layers",
    fontsize=14,
)

fig.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
import numpy as np
import matplotlib.pyplot as plt


# ---------------------------------------------------------
# Macro-averaged multiclass ROC curve
# ---------------------------------------------------------

def macro_roc_curve(y_true_onehot, scores):
    y_true_onehot = np.asarray(y_true_onehot)
    scores = np.asarray(scores)

    if y_true_onehot.shape != scores.shape:
        raise ValueError(
            "Targets and scores must have the same shape. "
            f"Targets: {y_true_onehot.shape}, "
            f"scores: {scores.shape}"
        )

    number_of_classes = y_true_onehot.shape[1]

    class_fprs = {}
    class_tprs = {}

    for class_index in range(number_of_classes):
        class_fprs[class_index], class_tprs[class_index], _ = (
            roc_curve(
                y_true_onehot[:, class_index],
                scores[:, class_index],
            )
        )

    # Shared FPR grid used to average the class-specific curves
    mean_fpr = np.linspace(0.0, 1.0, 1000)
    mean_tpr = np.zeros_like(mean_fpr)

    for class_index in range(number_of_classes):
        mean_tpr += np.interp(
            mean_fpr,
            class_fprs[class_index],
            class_tprs[class_index],
        )

    mean_tpr /= number_of_classes

    # Ensure exact ROC endpoints
    mean_tpr[0] = 0.0
    mean_tpr[-1] = 1.0

    macro_auc = auc(mean_fpr, mean_tpr)

    return mean_fpr, mean_tpr, macro_auc


# ---------------------------------------------------------
# Calculate ROC curves for the trained architectures
# ---------------------------------------------------------

architecture_roc_results = {}

for architecture_name, model in trained_architecture_models.items():

    layers = model["layers"]
    activation_funcs = model["activation_funcs"]

    # Softmax probabilities, shape: (n_test_samples, 10)
    test_scores = nnf.feed_forward(
        X_digits_test,
        layers,
        activation_funcs,
    )

    test_scores = np.asarray(test_scores)

    mean_fpr, mean_tpr, macro_auc = macro_roc_curve(
        Y_digits_test_onehot,
        test_scores,
    )

    architecture_roc_results[architecture_name] = {
        "False positive rate": mean_fpr,
        "True positive rate": mean_tpr,
        "AUC": macro_auc,
    }

    print(
        f"{architecture_name:<25} "
        f"macro-average AUC: {macro_auc:.4f}"
    )
# ---------------------------------------------------------
# Sort ROC results from highest to lowest AUC
# ---------------------------------------------------------

sorted_roc_results = sorted(
    architecture_roc_results.items(),
    key=lambda item: item[1]["AUC"],
    reverse=True,
)


# ---------------------------------------------------------
# Plot zoomed ROC curves
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 5.5))

for rank, (architecture_name, roc_result) in enumerate(
    sorted_roc_results,
    start=1,
):
    ax.plot(
        roc_result["False positive rate"],
        roc_result["True positive rate"],
        linewidth=2,
        label=(
            f"{rank}. {architecture_name} "
            f"(AUC = {roc_result['AUC']:.5f})"
        ),
    )


ax.set_title(
    "Macro-averaged ROC curves by architecture\n"
    "One-vs-rest classification on the digits dataset",
    fontsize=12,
)

ax.set_xlabel(
    "False positive rate",
    fontsize=11,
)

ax.set_ylabel(
    "True positive rate",
    fontsize=11,
)

# Zoom into the upper-left region
ax.set_xlim(0.0, 0.20)
ax.set_ylim(0.80, 1.005)

ax.tick_params(
    axis="both",
    labelsize=10,
)

ax.grid(
    alpha=0.25,
)

# Legend is ordered from highest to lowest AUC
ax.legend(
    loc="lower right",
    fontsize=10,
    framealpha=0.95,
)

fig.tight_layout()
plt.show()

Confusion matrices for Decreasing, constant deep and logistic:

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# ---------------------------------------------------------
# Architectures to include
# ---------------------------------------------------------

selected_architectures = [
    "Decreasing",
    "Constant deep",
    "Logistic regression",
]


# ---------------------------------------------------------
# Create confusion matrix plots
# ---------------------------------------------------------

fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 5),
)

for ax, architecture_name in zip(
    axes,
    selected_architectures,
):
    model = trained_architecture_models[
        architecture_name
    ]

    # Calculate softmax probabilities on the test set
    test_scores = nnf.feed_forward(
        X_digits_test,
        model["layers"],
        model["activation_funcs"],
    )

    test_scores = np.asarray(test_scores)

    # Select the class with the highest softmax probability
    predicted_labels = np.argmax(
        test_scores,
        axis=1,
    )

    # Plot confusion matrix
    ConfusionMatrixDisplay.from_predictions(
        y_digits_test,
        predicted_labels,
        labels=np.arange(10),
        display_labels=np.arange(10),
        cmap="Blues",
        colorbar=False,
        values_format="d",
        ax=ax,
    )

    test_accuracy = np.mean(
        predicted_labels == y_digits_test
    )

    ax.set_title(
        f"{architecture_name}\n"
        f"Test accuracy = {test_accuracy:.3f}",
        fontsize=11,
    )

    ax.set_xlabel("Predicted digit")
    ax.set_ylabel("True digit")


# ---------------------------------------------------------
# Overall title and layout
# ---------------------------------------------------------

fig.suptitle(
    "Confusion matrices by model architecture",
    fontsize=14,
)

fig.tight_layout()
plt.show()